# Social Media Multi-Agent System

In this session, you will be introduced to the foundational concepts of multi-agent systems and get an overview of the crewAI framework.

In [1]:
pip install crewai==1.14.1 crewai_tools==1.14.1 langchain_community==0.4.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 3.4 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opentelemetry-exporter-otlp-proto-grpc to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of opentelemetry-exporter-otlp-proto-grpc to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of typer to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━

- Import from the crewAI libray.

In [2]:
from crewai import Agent, Task, Crew
from crewai.tools import tool

In [3]:
from crewai_tools import SerperDevTool, ScrapeWebsiteTool, WebsiteSearchTool

- As a LLM for your agents, you'll be using OpenAI's `gpt-3.5-turbo`.

**Optional Note:** crewAI also allow other popular models to be used as a LLM for your Agents. You can see some of the examples at the [bottom of the notebook](#1).

In [ ]:
import openai

openai.api_key = " "

In [4]:
from google.colab import userdata


In [5]:
import os
#from utils import get_openai_api_key

#openai_api_key = get_openai_api_key()
import os
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["OPENAI_MODEL_NAME"] = 'gpt-4o-mini'

In [6]:
import os
os.environ["SERPER_API_KEY"] = userdata.get('SERPER_API_KEY')

## Creating Agents

- Define your Agents, and provide them a `role`, `goal` and `backstory`.
- It has been seen that LLMs perform better when they are role playing.

In [7]:
planner = Agent(
    role="Content Planner",
    goal="Plan engaging and factually accurate content on {topic}",
    backstory="You're working on planning a blog article "
              "about the topic: {topic}."
              "You collect information that helps the "
              "audience learn something "
              "and make informed decisions. "
              "Your work is the basis for "
              "the Content Writer to write an article on this topic.",
    tools=[SerperDevTool(), ScrapeWebsiteTool()],
    allow_delegation=False,
	verbose=True
)

### Agent: Writer

In [8]:
writer = Agent(
    role="Content Writer",
    goal="Write insightful and factually accurate "
         "opinion piece about the topic: {topic}",
    backstory="You're working on a writing "
              "a new opinion piece about the topic: {topic}. "
              "You base your writing on the work of "
              "the Content Planner, who provides an outline "
              "and relevant context about the topic. "
              "You follow the main objectives and "
              "direction of the outline, "
              "as provide by the Content Planner. "
              "You also provide objective and impartial insights "
              "and back them up with information "
              "provide by the Content Planner. "
              "You acknowledge in your opinion piece "
              "when your statements are opinions "
              "as opposed to objective statements.",
    tools=[SerperDevTool(), ScrapeWebsiteTool()],
    allow_delegation=False,
    verbose=True
)

### Agent: Editor

In [9]:
editor = Agent(
    role="Editor",
    goal="Edit a given blog post to align with "
         "the writing style of the organization. ",
    backstory="You are an editor who receives a blog post "
              "from the Content Writer. "
              "Your goal is to review the blog post "
              "to ensure that it follows journalistic best practices,"
              "provides balanced viewpoints "
              "when providing opinions or assertions, "
              "and also avoids major controversial topics "
              "or opinions when possible.",
    allow_delegation=False,
    verbose=True
)

## Creating Tasks

- Define your Tasks, and provide them a `description`, `expected_output` and `agent`.

### Task: Plan

In [10]:
plan = Task(
    description=(
        "1. Prioritize the latest trends, key players, "
            "and noteworthy news on {topic}.\n"
        "2. Identify the target audience, considering "
            "their interests and pain points.\n"
        "3. Develop a detailed content outline including "
            "an introduction, key points, and a call to action.\n"
        "4. Include SEO keywords and relevant data or sources."
    ),
    expected_output="A comprehensive content plan document "
        "with an outline, audience analysis, "
        "SEO keywords, and resources.",
    agent=planner,
)

### Task: Write

In [11]:
write = Task(
    description=(
        "1. Use the content plan to craft a compelling "
            "blog post on {topic}.\n"
        "2. Incorporate SEO keywords naturally.\n"
		"3. Sections/Subtitles are properly named "
            "in an engaging manner.\n"
        "4. Ensure the post is structured with an "
            "engaging introduction, insightful body, "
            "and a summarizing conclusion.\n"
        "5. Proofread for grammatical errors and "
            "alignment with the brand's voice.\n"
    ),
    expected_output="A well-written blog post "
        "in markdown format, ready for publication, "
        "each section should have 2 or 3 paragraphs.",
    agent=writer,
)

### Task: Edit

In [12]:
edit = Task(
    description=("Proofread the given blog post for "
                 "grammatical errors and "
                 "alignment with the brand's voice."),
    expected_output="A well-written blog post in markdown format, "
                    "ready for publication, "
                    "each section should have 2 or 3 paragraphs.",
    agent=editor
)

## Creating the Crew

- Create your crew of Agents
- Pass the tasks to be performed by those agents.
    - **Note**: *For this simple example*, the tasks will be performed sequentially (i.e they are dependent on each other), so the _order_ of the task in the list _matters_.
- `verbose=2` allows you to see all the logs of the execution.

In [14]:
crew = Crew(
    agents=[planner, writer, editor],
    tasks=[plan, write, edit],
    verbose=True
)

## Running the Crew

In [15]:
result = crew.kickoff(inputs={"topic": "Artificial Intelligence"})

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 319c20a0-4dce-4a69-a896-1456d3ebe312                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: 1. Prioritize the latest trends, key players, and noteworthy news on Artificial Intelligence.            │
│  2. Identify the target audience, considering their interests and pain points.                                  │
│  3. Develop a detailed content outline including an introduction, key points, and a call to action.             │
│  4. Include SEO keywords and relevant data or sources.                                                          │
│  ID: 668477cb-b544-43b6-9336-dcdcda079192                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Planner                                                                                         │
│                                                                                                                 │
│  Task: 1. Prioritize the latest trends, key players, and noteworthy news on Artificial Intelligence.            │
│  2. Identify the target audience, considering their interests and pain points.                                  │
│  3. Develop a detailed content outline including an introduction, key points, and a call to action.             │
│  4. Include SEO keywords and relevant data or sources.                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_the_internet_with_serper                                                                          │
│  Args: {'search_query': 'latest trends in AI 2023'}                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#3) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_the_internet_with_serper                                                                          │
│  Args: {'search_query': 'AI news 2023'}                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 1. Use the content plan to craft a compelling blog post on Artificial Intelligence.                      │
│  2. Incorporate SEO keywords naturally.                                                                         │
│  3. Sections/Subtitles are properly named in an engaging manner.                                                │
│  4. Ensure the post is structured with an engaging introduction, insightful body, and a summarizing             │
│  conclusion.                                                                                                    │
│  5. Proofread for grammatical errors and alignment with the brand's voice.                                      │
│                                                                                                                 │
│  Agent: Content Writer                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Proofread the given blog post for grammatical errors and alignment with the brand's voice.               │
│  ID: 5e4e2ebc-9e71-4911-88e2-79a296172c85                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Editor                                                                                                  │
│                                                                                                                 │
│  Task: Proofread the given blog post for grammatical errors and alignment with the brand's voice.               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Editor                                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```markdown                                                                                                    │
│  # Embracing the Future: Understanding the Landscape of Artificial Intelligence in 2023                         │
│                                                                                                                 │
│  The year 2023 has unveiled significant advancements in artificial intelligence (AI), heralding a new era of    │
│  innovation and transformation across multiple sectors. As generative AI technology gains traction, businesses  │
│  and organizations are eager to comprehend the implications of these advancements. Furthermore, the evolving    │
│  regulatory landscape necessitates a critical examination of the ethical usage of AI technologies, making it    │
│  essential for entrepreneurs, executives, researchers, and the general public to stay informed about current    │
│  trends and key players in the field.                                                                           │
│                                                                                                                 │
│  This article explores the latest trends, key players, and noteworthy news surrounding AI, emphasizing the      │
│  transformative impact AI is having on industries, the introduction of regulations, and the remarkable          │
│  companies and individuals driving the AI revolution.                                                           │
│                                                                                                                 │
│  ## Trend Analysis: The Rise of Generative AI                                                                   │
│                                                                                                                 │
│  2023 has witnessed a meteoric rise in generative AI, a technology that allows machines to create new content   │
│  based on existing data. From text and images to music and software code, generative AI is revolutionizing the  │
│  way content is produced and consumed. Businesses are leveraging this technology to enhance customer            │
│  engagement and increase productivity, resulting in improved user experiences. According to                     │
│  [McKinsey](https://www.mckinsey.com/capabilities/quantumblack/our-insights/the-state-of-ai-in-2023-generative  │
│  -ais-breakout-year), generative AI's integration into various functions has led to measurable outcomes in      │
│  performance and innovation.                                                                                    │
│                                                                                                                 │
│  Moreover, industries such as healthcare and finance are capitalizing on AI advancements. Predictive            │
│  analytics, a subset of AI, is helping healthcare providers analyze patient data more effectively to deliver    │
│  personalized medicine, while finance professionals utilize AI for risk assessments and fraud detection,        │
│  ultimately enhancing decision-making processes. The widespread adoption of AI showcases not only the           │
│  technology's versatility but also its crucial role in shaping the future of work and interaction.              │
│                                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Proofread the given blog post for grammatical errors and alignment with the brand's voice.               │
│  Agent: Editor                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

- Display the results of your execution as markdown in the notebook.

In [16]:
from IPython.display import Markdown, Image, display

caption = result.raw
image_url = result.tasks_output[0].raw

display(Image(url=image_url))
display(Markdown(caption))

```markdown
# Embracing the Future: Understanding the Landscape of Artificial Intelligence in 2023

The year 2023 has unveiled significant advancements in artificial intelligence (AI), heralding a new era of innovation and transformation across multiple sectors. As generative AI technology gains traction, businesses and organizations are eager to comprehend the implications of these advancements. Furthermore, the evolving regulatory landscape necessitates a critical examination of the ethical usage of AI technologies, making it essential for entrepreneurs, executives, researchers, and the general public to stay informed about current trends and key players in the field. 

This article explores the latest trends, key players, and noteworthy news surrounding AI, emphasizing the transformative impact AI is having on industries, the introduction of regulations, and the remarkable companies and individuals driving the AI revolution.

## Trend Analysis: The Rise of Generative AI

2023 has witnessed a meteoric rise in generative AI, a technology that allows machines to create new content based on existing data. From text and images to music and software code, generative AI is revolutionizing the way content is produced and consumed. Businesses are leveraging this technology to enhance customer engagement and increase productivity, resulting in improved user experiences. According to [McKinsey](https://www.mckinsey.com/capabilities/quantumblack/our-insights/the-state-of-ai-in-2023-generative-ais-breakout-year), generative AI's integration into various functions has led to measurable outcomes in performance and innovation. 

Moreover, industries such as healthcare and finance are capitalizing on AI advancements. Predictive analytics, a subset of AI, is helping healthcare providers analyze patient data more effectively to deliver personalized medicine, while finance professionals utilize AI for risk assessments and fraud detection, ultimately enhancing decision-making processes. The widespread adoption of AI showcases not only the technology's versatility but also its crucial role in shaping the future of work and interaction.

## Navigating the Regulatory Landscape

As AI technologies continue to evolve, so does the recognition of their potential ethical implications. In response to these challenges, the European Union has introduced new regulations focused on AI technology, aiming to ensure ethical use and risk management. These regulatory frameworks are designed to protect consumers while promoting innovation, as outlined in [ODSC's](https://odsc.medium.com/the-biggest-ai-news-that-shook-the-world-in-2023-3aa925b84343) summary of AI developments in 2023. Businesses must now adapt to this shifting landscape, considering compliance as an integral aspect of implementing AI solutions.

The regulatory landscape emphasizes the importance of transparency and accountability in AI deployment. Companies must increasingly prioritize ethical considerations by investing in responsible AI practices, thereby fostering trust among consumers and stakeholders. As AI adoption continues to rise, navigating the complex intersection of law, technology, and ethics will be paramount for businesses aiming to thrive in the market.

## Major Players: Leaders Paving the Way

The AI industry is marked by several key players whose contributions are shaping the future of technology. NVIDIA stands out as a leading provider of Graphics Processing Units (GPUs), essential for accelerating AI advancements. OpenAI, known for creating the famous ChatGPT, has significantly influenced natural language processing, setting standards for conversational AI. Furthermore, tech giants like Google and Microsoft are maintaining their edge through continuous AI research and innovation, integrating AI into various products and services.

Notably, influential individuals such as Sam Altman (OpenAI), Elon Musk (xAI), and Demis Hassabis (DeepMind) are at the forefront of AI’s evolution, driving thought-provoking discussions surrounding technology’s future. By recognizing the vital contributions of these companies and leaders, stakeholders can better understand the dynamics shaping the AI landscape and explore collaboration opportunities.

## Real-World Applications: AI in Action

Numerous case studies demonstrate the profound impact of AI across various sectors. For instance, healthcare organizations are implementing AI-driven solutions for diagnostic imaging and patient management, yielding improved healthcare outcomes. In the automotive industry, AI is optimizing supply chain logistics, resulting in lower costs and increased efficiency. Tailoring solutions based on AI applications not only drives operational excellence but also fosters innovation, leading to better products and services for consumers. 

Furthermore, as businesses increasingly adopt AI technologies, they must remain cognizant of potential challenges, such as integrating AI into legacy systems and managing workforce transitions. By addressing these challenges, organizations can seamlessly incorporate AI into their existing frameworks, enhancing productivity and fostering growth.

## Future Outlook: What Lies Ahead

The trajectory of AI development suggests a landscape teeming with possibilities. As we look towards the future, the pace of innovation is expected to accelerate, with advancements in AI models and applications driving further disruption across industries. Investment in generative AI businesses is projected to continue growing by approximately 30%, underscoring the ongoing interest in AI technologies and their potential to revolutionize business practices. 

In conclusion, the unfolding narrative of artificial intelligence in 2023 serves as a reminder of the potential this technology has to transform our lives, organizations, and society. By staying informed about trends, leveraging key players' advancements, and adhering to regulations, stakeholders can navigate the AI landscape more effectively. Encouraging curiosity and awareness will be essential in harnessing AI's capabilities, empowering all to consider how they can use AI to benefit their personal and professional endeavors.

---

As we enter this new era of AI, let’s stay informed and engaged in discussions that shape our understanding and use of this transformative technology. The future is bright, and it's one worth exploring together.
```